In [ ]:
import pandas as pd
import geopandas as gpd
from datetime import datetime as dt
import numpy as np
import math
import glob
import re
import os
import shapely.wkb
from shapely import wkt, wkb
# import statsmodels.api as sm
# import matplotlib.pyplot as plt
# from sklearn.cluster import KMeans
from math import ceil
pd.options.display.max_columns = None

In [ ]:
# Cargar archivos
feather_files = glob.glob(os.path.join("..", "jams_dim", "*.parquet"))
borde_oriental = gpd.read_file("../outputs/borde_oriental.shp")

# Parámetros
n_chunks = 10
chunk_size = ceil(len(feather_files) / n_chunks)
filtrar = False

# Leer malla vial y preparar buffers
mvi = gpd.read_file(r"../estructura_vial/MVI.geojson")
mvi.to_crs(4326, inplace=True)
mvi.columns = mvi.columns.str.lower()
mvi["id_mvi"] = mvi["mviciv"].astype(str) + "_" + mvi["mvisvia"].str.replace(" ", "")
mvi['contador'] = mvi.groupby('id_mvi').cumcount() + 1
mvi['id_mvi'] = mvi['id_mvi'] + '.' + mvi['contador'].astype(str)

buffer_distance = 4
mvi_buffered = mvi.to_crs("EPSG:3116").copy()
mvi_buffered['mvi_length'] = mvi_buffered.geometry.length
mvi_buffered['buffer_geom'] = mvi_buffered.geometry.buffer(buffer_distance, cap_style="square")
mvi_buffered_buffer = mvi_buffered.set_geometry('buffer_geom')
mvi_buffered_buffer = mvi_buffered_buffer.to_crs("EPSG:3116")

# Crear carpeta de salida si no existe
os.makedirs("outputs", exist_ok=True)

In [ ]:
# Proceso por chunks
for i in range(n_chunks):
    print(f"Procesando chunk {i+1}/{n_chunks}...")
    gdfs_filtrados = []
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(feather_files))

    for archivo in feather_files[start_idx:end_idx]:
        df = pd.read_parquet(archivo)
        df['geometry'] = df['geom_wkt'].apply(
            lambda geom: wkb.loads(geom) if isinstance(geom, bytes)
            else wkt.loads(geom) if isinstance(geom, str)
            else geom
        )
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=4326)

        if filtrar:
            filtro = gdf.within(borde_oriental.iloc[0]['geometry'])
            gdf_filtrado = gdf[filtro].copy()
        else:
            gdf_filtrado = gdf.copy()

        gdf_filtrado['timestamp_pubMillis'] = pd.to_datetime(gdf_filtrado['pubMillis_ts_cot'], unit='ms')
        gdf_filtrado['timestamp_startTime'] = pd.to_datetime(gdf_filtrado['startTime_ts_cot'], unit='ms')
        gdf_filtrado['timestamp_endTime'] = pd.to_datetime(gdf_filtrado['endTime_ts_cot'], unit='ms')

        gdf_filtrado = gdf_filtrado[['uuid', 'timestamp_pubMillis', 'timestamp_startTime', 'timestamp_endTime',
                                     'speedKMH', 'level', 'length', 'delay', 'geometry']]
        gdfs_filtrados.append(gdf_filtrado)

    if not gdfs_filtrados:
        continue

    jams = gpd.GeoDataFrame(pd.concat(gdfs_filtrados, ignore_index=True), crs=4326)
    jams["jam_id"] = jams.index

    grouped_jams = jams.groupby("uuid").agg({
        'timestamp_startTime': 'min',
        'timestamp_endTime': 'max',
        'jam_id': 'count'
    }).reset_index()
    grouped_jams.columns = ['uuid', 'general_timestamp_startTime', 'general_timestamp_endTime', 'jam_id_counts']

    jams = jams[jams["delay"] != -1]  ## Esto de puede cambiar si no interesa que se eliminen los jams en vias cerradas.
    jams = jams.merge(grouped_jams, how="left", on='uuid')

    jams = jams.to_crs("EPSG:3116")

    # Aplicar overlay pesado dentro del for y guardar
    jams_intersections = gpd.overlay(jams, mvi_buffered_buffer[['id_mvi', 'mvi_length', 'buffer_geom']], how='intersection')
    jams_intersections['intersection_length'] = jams_intersections.geometry.length
    jams_intersections['overlap_ratio'] = jams_intersections['intersection_length'] / jams_intersections['mvi_length']
    jams_intersections = jams_intersections[jams_intersections['overlap_ratio'] > 0.2]

    jams_intersections["timeframe"] = jams_intersections["timestamp_pubMillis"].dt.floor("15min")
    jams_intersections['delay'] = jams_intersections['delay'] * (jams_intersections['intersection_length'] / jams_intersections['length'])

    # Guardar resultado del overlay intermedio para cada chunk
    overlay_output_path = f"outputs/jams_intersections_chunk_{i+1}.parquet"
    jams_intersections.to_parquet(overlay_output_path, index=False)
    print(f"Overlay del chunk {i+1} guardado en {overlay_output_path}")

In [ ]:
## Lectura, agregación y agrupación
# === Fase final: Leer todos los resultados y hacer la agregación global ===
print("\nIniciando agregación global de los resultados...")
all_intersections_files = sorted(glob.glob("outputs/jams_intersections_chunk_*.parquet"))
all_dfs = [pd.read_parquet(f) for f in all_intersections_files]

combined_df = pd.concat(all_dfs, ignore_index=True)

aggregated = combined_df.groupby(["id_mvi", "timeframe"]).agg({
    "speedKMH": ["min", "mean", "max"],
    "level": ["min", "mean", "max"],
    "delay": ["min", "sum", "max"],
    "jam_id": "count"
}).reset_index()

aggregated.columns = [
    "id_mvi", "timeframe",
    "vel_min", "vel_mean", "vel_max",
    "level_min", "level_mean", "level_max",
    "delay_min", "delay_sum", "delay_max",
    "count_jams"
]

aggregated.sort_values(by=["id_mvi", "timeframe"], inplace=True)

# Guardar resultado final
aggregated.to_parquet("outputs/jams_intersections_aggregated.parquet", index=False)
print("Agregación global completada y guardada en 'outputs/jams_intersections_aggregated.parquet'")